In [62]:
import requests
from bs4 import BeautifulSoup
import nltk
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import ssl
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score

import sys
sys.path.append('KeyClass/keyclass/')
sys.path.append('KeyClass/scripts/')

import gc
import argparse
import label_data, encode_datasets, train_downstream_model
import torch
import pickle
import numpy as np
import os
from os.path import join, exists
from datetime import datetime
import utils
import models
import create_lfs
import train_classifier
import importlib

random_seed = 0 # Random seed for experiments

In [2]:
nltk.download("stopwords")
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Jon\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Jon\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Jon\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
# Build our class descriptions table.

# Scrape codes from wikipedia.
icd_9_codes_wiki = requests.get("https://en.wikipedia.org/wiki/List_of_ICD-9_codes_E_and_V_codes:_external_causes_of_injury_and_supplemental_classification")
soup = BeautifulSoup(icd_9_codes_wiki.text, "html.parser")

In [4]:
tds_with_nowrap = soup.find_all('td')

code_mapping = dict()
category_descs_concat = dict()

for td in tds_with_nowrap:
    span = td.find('span', class_='nowrap')
    if span:
        try:
            split = span.a['title'].split(":")
            title = split[1][1:]
            codes = split[0]
            codes = codes.split(" ")[4].split("–")
            for i in range(int(codes[0]), int(codes[1]) + 1):
                code_mapping[i] = title
            category_descs_concat[title] = ""
        except:
            pass

external_str = "external causes of injury"
supp_str = "supplementary"
category_descs_concat[external_str] = ""
category_descs_concat[supp_str] = ""

In [5]:
#Descriptions are pulled from cms.gov
with open('data/CMS32_DESC_LONG_DX.txt', 'r', encoding='latin1') as file:
    lines = [line.strip() for line in file]

In [6]:
for line in lines:
    split = line.split(" ")
    code = split[0][:3]
    desc = " ".join(split[2:])
    if code[0] == "E":
        category = external_str
    elif code[0] == "V":
        category = supp_str
    else:
        category = code_mapping[int(code)]
    category_descs_concat[category] += desc + " "

In [7]:
stop_words = set(stopwords.words('english'))

In [8]:
final_output = dict()
num_most_common = 20


for category in category_descs_concat:
    concat = category_descs_concat[category]
    tokenized = words = word_tokenize(concat.lower())
    tokenized = [word for word in tokenized if word not in stop_words and word not in string.punctuation]
    word_freq = Counter(tokenized)
    most_common = word_freq.most_common(num_most_common)
    most_common_lst = [word for word, _ in most_common]
    most_common_str = ' '.join(most_common_lst)
    final_output[category] = most_common_str

In [9]:
df = pd.DataFrame.from_dict(final_output, orient="index")

if not os.path.exists("data"):
    os.mkdir("data")
df.to_csv("data/mined_descriptions.csv")

In [10]:
# Generate targets for the config file
count = 0
for index, row in df.iterrows():
    if count < 10:
        target_name = "target_" + "0" + str(count)
    else:
        target_name = "target_" + str(count)
    val_str = row[0].replace(' ', ', ')
    print(target_name + ": " + val_str)
    count += 1

target_00: found, unspecified, bacilli, bacteriological, examination, tubercle, tuberculosis, histological, specified, sputum, microscopy, confirmed, infection, due, bacterial, done, unknown, present, culture, histologically
target_01: neoplasm, malignant, lymph, nodes, unspecified, sites, benign, cell, limb, site, lymphoma, skin, leukemia, specified, carcinoma, disease, tumor, remission, upper, lower
target_02: type, unspecified, uncontrolled, disorders, deficiency, manifestations, stated, metabolism, specified, mention, mellitus, diabetes, ii, juvenile, without, goiter, disorder, vitamin, thyrotoxic, crisis
target_03: unspecified, anemia, disease, deficiency, specified, anemias, blood, crisis, hemolytic, thalassemia, iron, hereditary, due, thrombocytopenia, secondary, chronic, without, cell, congenital, factor
target_04: disorder, unspecified, type, episode, remission, dependence, schizophrenia, abuse, specified, drug, disorders, current, acute, episodic, recent, depressive, psychoti

In [11]:
# Read the MIMIC-III data
icd = pd.read_csv('data/DIAGNOSES_ICD.csv')
notes = pd.read_csv('data/NOTEEVENTS.csv')

C:\Users\Jon\AppData\Local\Temp\ipykernel_34508\2024626850.py:3: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  notes = pd.read_csv('data/NOTEEVENTS.csv')


In [12]:
merged = pd.merge(notes, icd, on=["SUBJECT_ID", "HADM_ID"])
merged = merged.dropna(subset=['TEXT', 'ICD9_CODE'])
merged = merged[merged['CATEGORY'] == 'Discharge summary']

In [13]:
category_to_number_mapping = {}
count = 0
for key in final_output:
    category_to_number_mapping[key] = count
    count += 1
def map_labels(icd9_code):
    code = icd9_code[:3]
    if code[0] == "E":
        category = external_str
    elif code[0] == "V":
        category = supp_str
    else:
        category = code_mapping[int(code)]
    return category_to_number_mapping[category]

In [14]:
merged['label'] = merged['ICD9_CODE'].apply(map_labels)

In [15]:
data = merged[['TEXT', 'label']]
data = data.drop_duplicates(subset=['TEXT', 'label'])

In [70]:
category_to_number_mapping

{'infectious and parasitic diseases': 0,
 'neoplasms': 1,
 'endocrine, nutritional and metabolic diseases, and immunity disorders': 2,
 'diseases of the blood and blood-forming organs': 3,
 'mental disorders': 4,
 'diseases of the nervous system and sense organs': 5,
 'diseases of the circulatory system': 6,
 'diseases of the respiratory system': 7,
 'diseases of the digestive system': 8,
 'diseases of the genitourinary system': 9,
 'complications of pregnancy, childbirth, and the puerperium': 10,
 'diseases of the skin and subcutaneous tissue': 11,
 'diseases of the musculoskeletal system and connective tissue': 12,
 'congenital anomalies': 13,
 'certain conditions originating in the perinatal period': 14,
 'symptoms, signs, and ill-defined conditions': 15,
 'injury and poisoning': 16,
 'external causes of injury': 17,
 'supplementary': 18}

In [16]:
# Encode the data
encoded_df_dict = {}
for index, row in data.iterrows():
    if row['TEXT'] not in encoded_df_dict:
        encoded_df_dict[row['TEXT']] = [row['label']]
    else:
        encoded_df_dict[row['TEXT']].append(row['label'])

In [17]:
for key in encoded_df_dict:
    val = encoded_df_dict[key]
    encoded_df_dict[key] = ' '.join(str(x) for x in encoded_df_dict[key])
encoded_df = pd.DataFrame.from_dict(encoded_df_dict, orient='index').reset_index()
encoded_df.columns = ['text', 'labels']

In [18]:
# randomly sample 10% of the data to use (due to computational limits)
sampled_df = encoded_df.sample(frac=1, random_state=random_seed)

In [19]:
# Use TF-IDF to get the most relevant words in the text
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(sampled_df['text'])

In [20]:
feature_names = np.array(vectorizer.get_feature_names())
top_k = 50
new_texts = []

for doc_idx in range(tfidf_matrix.shape[0]):
    row = tfidf_matrix.getrow(doc_idx)
    row_data = row.data
    row_indices = row.indices
    
    # Sort indices by TF-IDF score
    top_indices = row_indices[np.argsort(row_data)[::-1][:top_k]]
    
    # Get corresponding words
    top_words = feature_names[top_indices]
    
    # Join them back into a document
    new_texts.append(' '.join(top_words))

# 3. Replace or add a new column
sampled_df['text'] = new_texts


In [21]:
# Write the table into train/split .txt files.
train_data, test_data = train_test_split(sampled_df, test_size=0.3, random_state=random_seed)

In [22]:
if not os.path.exists("data/mimic"):
    os.mkdir("data/mimic")

with open('data/mimic/train.txt', 'w', encoding='utf-8') as f_text, \
     open('data/mimic/train_labels.txt', 'w', encoding='utf-8') as f_label:
    for text, label in zip(train_data['text'], train_data['labels']):
        f_text.write(text.strip().replace('\n', ' ') + '\n')
        f_label.write(str(label) + '\n')

In [23]:
with open('data/mimic/test.txt', 'w', encoding='utf-8') as f_text, \
     open('data/mimic/test_labels.txt', 'w', encoding='utf-8') as f_label:
    for text, label in zip(test_data['text'], test_data['labels']):
        f_text.write(text.strip().replace('\n', ' ') + '\n')
        f_label.write(str(label) + '\n')

In [3]:
# Input arguments
config_file_path = r'mimic_config.yml' # Specify path to the configuration file
default_config_path = r'KeyClass/config_files/default_config.yml'

# Encode the dataset

In [4]:
# importlib.reload(utils)

args = utils.Parser(config_file_path=config_file_path, default_config_file_path=default_config_path).parse()

if args['use_custom_encoder']:
    model = models.CustomEncoder(pretrained_model_name_or_path=args['base_encoder'], 
        device='cuda' if torch.cuda.is_available() else 'cpu')
else:
    model = models.Encoder(model_name=args['base_encoder'], 
        device='cuda' if torch.cuda.is_available() else 'cpu')

for split in ['train', 'test']:
    sentences = utils.fetch_data(dataset=args['dataset'], split=split, path=args['data_path'])
    embeddings = model.encode(sentences=sentences, batch_size=args['end_model_batch_size'], 
                                show_progress_bar=args['show_progress_bar'], 
                                normalize_embeddings=args['normalize_embeddings'])
    with open(join(args['data_path'], args['dataset'], f'{split}_embeddings.pkl'), 'wb') as f:
        pickle.dump(embeddings, f)


Some weights of the model checkpoint at bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12 were not used when initializing BertModel: ['cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Batches:   0%|          | 0/326 [00:00<?, ?it/s]

Batches:   0%|          | 0/140 [00:00<?, ?it/s]

In [5]:
# Probabilistically label the data

args = utils.Parser(config_file_path=config_file_path, default_config_file_path=default_config_path).parse()

# Load training data
train_text = utils.fetch_data(dataset=args['dataset'], path=args['data_path'], split='train')

training_labels_present = False
if exists(join(args['data_path'], args['dataset'], 'train_labels.txt')):
    with open(join(args['data_path'], args['dataset'], 'train_labels.txt'), 'r') as f:
        y_train = f.readlines()
    # y_train = np.array([int(i.replace('\n','')) for i in y_train])
    y_train = np.array([[int(i) for i in labels.strip().split()] for labels in y_train], dtype=object)
    training_labels_present = True
else:
    y_train = None
    training_labels_present = False
    print('No training labels found!')

with open(join(args['data_path'], args['dataset'], 'train_embeddings.pkl'), 'rb') as f:
    X_train = pickle.load(f)

# Change the target to 19-dimensional one-hot vectors as stated in the paper
binarizer = MultiLabelBinarizer()
y_train_encoded = binarizer.fit_transform(y_train)

# Print dataset statistics
print(f"Getting labels for the {args['dataset']} data...")
print(f'Size of the data: {len(train_text)}')
if training_labels_present:
    print('Class distribution', np.unique(np.hstack(y_train), return_counts=True))

class_balance = np.unique(np.hstack(y_train), return_counts=True)[1] / np.unique(np.hstack(y_train), return_counts=True)[1].sum()

Getting labels for the mimic data...
Size of the data: 41693
Class distribution (array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18]), array([11272,  6643, 27263, 14881, 12311, 12010, 32847, 19785, 15972,
       16841,   122,  4731,  7646,  2225,  2907, 15316, 17529, 12496,
       22714], dtype=int64))


In [81]:
class_balance

array([0.04411552, 0.02599888, 0.10669991, 0.05824015, 0.04818188,
       0.04700385, 0.12855415, 0.07743307, 0.06251003, 0.06591106,
       0.00047747, 0.01851584, 0.02992435, 0.00870804, 0.0113772 ,
       0.05994262, 0.0686037 , 0.04890592, 0.08889637])

In [7]:
y_train_encoded[1]

array([0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1])

In [8]:
# Creating labeling functions

importlib.reload(create_lfs)
args = utils.Parser(config_file_path=config_file_path, default_config_file_path=default_config_path).parse()

# Load label names/descriptions
label_names = []
for a in args:
    if 'target' in a: label_names.append(args[a])

labeler = create_lfs.CreateLabellingFunctions(base_encoder=args['base_encoder'], 
                                            device=torch.device(args['device']),
                                            label_model=args['label_model'])

print("finished creating labelling functions.")

proba_preds = labeler.get_labels(text_corpus=train_text, label_names=label_names, min_df=args['min_df'], 
                                ngram_range=args['ngram_range'], topk=args['topk'], y_train=y_train_encoded, 
                                label_model_lr=args['label_model_lr'], label_model_n_epochs=args['label_model_n_epochs'], 
                                verbose=True, n_classes=args['n_classes'], class_balance=class_balance)


# Save the predictions
if not os.path.exists(args['preds_path']): os.makedirs(args['preds_path'])
with open(join(args['preds_path'], f"{args['label_model']}_proba_preds.pkl"), 'wb') as f:
    pickle.dump(proba_preds, f)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Jon\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Some weights of the model checkpoint at bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12 were not used when initializing BertModel: ['cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequ

finished creating labelling functions.
Found assigned category counts [ 160   44   17   45   94   41   72  173  117  217  834  121   81   24
  137 1047   28   52 1073]
labeler.vocabulary:
 4377
labeler.word_indicator_matrix.shape (41693, 323)
Len keywords 323
assigned_category: Unique and Counts (array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18], dtype=int64), array([17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17,
       17, 17], dtype=int64))
found, unspecified, bacilli, bacteriological, examination, tubercle, tuberculosis, histological, specified, sputum, microscopy, confirmed, infection, due, bacterial, done, unknown, present, culture, histologically ['aspergillus' 'cavitary' 'cholestyramine' 'clostridium' 'diflucan'
 'dinitrate' 'enterococcal' 'enterococcus' 'intraoperative'
 'intraoperatively' 'nystatin' 'postoperative' 'postoperatively'
 'pseudomonal' 'tegretol' 'tobramycin' 'viridans']
neoplasm, malignant, lymph, nodes,

INFO:root:Computing O...
INFO:root:Estimating \mu...
INFO:root:Using GPU...
100%|█████████████████████████████████████████████████████████████████████████████| 100/100 [01:26<00:00,  1.16epoch/s]
INFO:root:Finished Training


In [24]:
importlib.reload(train_downstream_model)
importlib.reload(models)

torch.manual_seed(random_seed)
np.random.seed(random_seed)

# Load data
X_train_embed_masked, y_train_lm_masked, y_train_masked, \
	X_test_embed, y_test, training_labels_present, \
	sample_weights_masked, proba_preds_masked = train_downstream_model.load_data(args, class_balance=class_balance)


Confidence of least confident data point of class 0: 0.6083392684676318
Confidence of least confident data point of class 1: 0.9867820964659001
Confidence of least confident data point of class 2: 0.662365202804242
Confidence of least confident data point of class 3: 0.6189325708978851
Confidence of least confident data point of class 4: 0.6119019760063481
Confidence of least confident data point of class 5: 0.6084623246057411
Confidence of least confident data point of class 6: 0.7237307997362331
Confidence of least confident data point of class 7: 0.6333750979431361
Confidence of least confident data point of class 8: 0.6163800147469776
Confidence of least confident data point of class 9: 0.6089292657802582
Confidence of least confident data point of class 10: 0.9999870223431964
Confidence of least confident data point of class 11: 0.6019160870849535
Confidence of least confident data point of class 12: 0.5989137258906237
Confidence of least confident data point of class 13: 0.995603

In [25]:
# Train a downstream classifier
importlib.reload(train_classifier)
importlib.reload(train_downstream_model)

args = utils.Parser(config_file_path=config_file_path, default_config_file_path=default_config_path).parse()

if args['use_custom_encoder']:
	encoder = models.CustomEncoder(pretrained_model_name_or_path=args['base_encoder'], device=args['device'])
else:
	encoder = models.Encoder(model_name=args['base_encoder'], device=args['device'])

classifier = models.FeedForwardFlexible(encoder_model=encoder,
										h_sizes=args['h_sizes'], 
										activation=eval(args['activation']),
										device=torch.device(args['device']))
print('\n===== Training the downstream classifier =====\n')
model = train_classifier.train(model=classifier, 
							device=torch.device(args['device']),
							X_train=X_train_embed_masked, 
							y_train=y_train_lm_masked,
							sample_weights=sample_weights_masked if args['use_noise_aware_loss'] else None, 
							epochs=args['end_model_epochs'], 
							batch_size=args['end_model_batch_size'], 
							criterion=eval(args['criterion']), 
							raw_text=False, 
							lr=eval(args['end_model_lr']), 
							weight_decay=eval(args['end_model_weight_decay']),
							patience=args['end_model_patience'])


end_model_preds_train = model.predict_proba(torch.from_numpy(X_train_embed_masked), batch_size=512, raw_text=False)
end_model_preds_test = model.predict_proba(torch.from_numpy(X_test_embed), batch_size=512, raw_text=False)

Some weights of the model checkpoint at bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12 were not used when initializing BertModel: ['cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



===== Training the downstream classifier =====



Epoch 19: 100%|█████████████| 20/20 [00:03<00:00,  6.46batch/s, best_loss=0.759, running_loss=0.758, tolerance_count=0]


In [26]:
# Self train the classifier
importlib.reload(train_classifier)

args = utils.Parser(config_file_path=config_file_path, default_config_file_path=default_config_path).parse()

with open(
        join(args['data_path'], args['dataset'], f'train_embeddings.pkl'),
        'rb') as f:
    X_train_embed = pickle.load(f)
with open(join(args['data_path'], args['dataset'], f'test_embeddings.pkl'),
          'rb') as f:
    X_test_embed = pickle.load(f)

model = train_classifier.self_train(model=model, 
									X_train=X_train_embed, 
									X_val=X_test_embed, 
									y_val=y_test, 
									device=torch.device(args['device']), 
									lr=eval(args['self_train_lr']), 
									weight_decay=eval(args['self_train_weight_decay']),
									patience=args['self_train_patience'], 
									batch_size=args['self_train_batch_size'], 
									q_update_interval=args['q_update_interval'],
									self_train_thresh=eval(args['self_train_thresh']), 
									print_eval=True, raw_text=False)

Epoch 2:  33%|██    | 2/6 [00:01<00:02,  1.65batch/s, self_train_agreement=1, tolerance_count=2, validation_accuracy=0]


In [27]:
model.eval()

FeedForwardFlexible(
  (encoder_model): CustomEncoder(
    (model): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0): BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, bias=True

In [28]:
end_model_preds_test = model.predict_proba(torch.from_numpy(X_test_embed), batch_size=args['self_train_batch_size'], raw_text=False)

In [49]:
y_test_encoded = binarizer.fit_transform(y_test)

testing_metrics = utils.compute_metrics_bootstrap(y_preds=(end_model_preds_test>0.46).astype(int),
													y_true=y_test_encoded, 
													average=args['average'], 
													n_bootstrap=args['n_bootstrap'], 
													n_jobs=args['n_jobs'])

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  40 tasks      | elapsed:    0.8s
[Parallel(n_jobs=10)]: Done 100 out of 100 | elapsed:    1.9s finished


In [74]:
for i in range(len(testing_metrics)):
    if i == 0:
        print("Accuracy (mean, std): ", testing_metrics[i])
    elif i == 1:
        print("Precision (mean, std): ", testing_metrics[i])
    elif i == 2:
        print("Recall (mean, std): ", testing_metrics[i])
    else:
        print("F1 Score (mean, std): ", testing_metrics[i])

Accuracy (mean, std):  [0.65993891 0.00069918]
Precision (mean, std):  [0.46695915 0.04370757]
Recall (mean, std):  [0.50630315 0.00116584]
F1 Score (mean, std):  [0.48472997 0.02462871]


In [65]:
def calc_class_f1s(y_pred, y_true, avg='weighted'):
    class_y_preds = {}
    class_y_trues = {}
    for i in range(len(y_pred[0])):
        class_y_preds[i] = []
        class_y_trues[i] = []

    for i in range(len(y_pred)):
        y_pred_cur = y_pred[i]
        y_true_cur = y_true[i]
        for j in range(len(y_pred_cur)):
            class_y_preds[j].append(y_pred_cur[j])
            class_y_trues[j].append(y_true_cur[j])

    f1s = {}
    for idx in class_y_preds:
        precision = precision_score(class_y_trues[idx], class_y_preds[idx], average=avg)
        recall = recall_score(class_y_trues[idx], class_y_preds[idx], average=avg)
        f1s[idx] = 2 * (precision * recall) / (precision + recall)
    return f1s

In [66]:
f1_scores = calc_class_f1s((end_model_preds_test>0.46).astype(int), y_test_encoded)

C:\Users\Jon\anaconda3\envs\keyclass\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Jon\anaconda3\envs\keyclass\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Jon\anaconda3\envs\keyclass\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [68]:
f1_scores

{0: 0.11609765152200162,
 1: 0.7802761767349686,
 2: 0.42759949151836973,
 3: 0.45292688667051234,
 4: 0.7521215065667447,
 5: 0.5952619612971619,
 6: 0.5964060926027296,
 7: 0.5300391972434956,
 8: 0.5889818532698258,
 9: 0.466855650772983,
 10: 0.9654194575946644,
 11: 0.462113375195023,
 12: 0.7368199497191452,
 13: 0.005389654784976455,
 14: 0.895737789163026,
 15: 0.5751483268016218,
 16: 0.24311193100998232,
 17: 0.3968175987399254,
 18: 0.5056732606972453}

In [82]:
end_model_preds_test[3]

array([0.45836812, 0.45813778, 0.46343863, 0.45843095, 0.45783487,
       0.4510485 , 0.47374135, 0.46952108, 0.44529048, 0.46254995,
       0.4444903 , 0.46167758, 0.45664293, 0.46031916, 0.45309317,
       0.45839208, 0.44806144, 0.46878383, 0.4668068 ], dtype=float32)

In [69]:
# Ablation - just skip self training
# Train a downstream classifier
importlib.reload(train_classifier)
importlib.reload(train_downstream_model)

args = utils.Parser(config_file_path=config_file_path, default_config_file_path=default_config_path).parse()

if args['use_custom_encoder']:
	encoder = models.CustomEncoder(pretrained_model_name_or_path=args['base_encoder'], device=args['device'])
else:
	encoder = models.Encoder(model_name=args['base_encoder'], device=args['device'])

classifier = models.FeedForwardFlexible(encoder_model=encoder,
										h_sizes=args['h_sizes'], 
										activation=eval(args['activation']),
										device=torch.device(args['device']))
print('\n===== Training the downstream classifier =====\n')
model = train_classifier.train(model=classifier, 
							device=torch.device(args['device']),
							X_train=X_train_embed_masked, 
							y_train=y_train_lm_masked,
							sample_weights=sample_weights_masked if args['use_noise_aware_loss'] else None, 
							epochs=args['end_model_epochs'], 
							batch_size=args['end_model_batch_size'], 
							criterion=eval(args['criterion']), 
							raw_text=False, 
							lr=eval(args['end_model_lr']), 
							weight_decay=eval(args['end_model_weight_decay']),
							patience=args['end_model_patience'])


end_model_preds_train = model.predict_proba(torch.from_numpy(X_train_embed_masked), batch_size=512, raw_text=False)
end_model_preds_test = model.predict_proba(torch.from_numpy(X_test_embed), batch_size=512, raw_text=False)

Some weights of the model checkpoint at bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12 were not used when initializing BertModel: ['cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



===== Training the downstream classifier =====



Epoch 19: 100%|█████████████| 20/20 [00:02<00:00,  7.09batch/s, best_loss=0.759, running_loss=0.757, tolerance_count=0]


In [70]:
testing_metrics = utils.compute_metrics_bootstrap(y_preds=(end_model_preds_test>0.46).astype(int),
													y_true=y_test_encoded, 
													average=args['average'], 
													n_bootstrap=args['n_bootstrap'], 
													n_jobs=args['n_jobs'])

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    4.8s
[Parallel(n_jobs=10)]: Done 100 out of 100 | elapsed:    6.0s finished


In [71]:
testing_metrics

array([[0.65993891, 0.00069918],
       [0.46695915, 0.04370757],
       [0.50630315, 0.00116584],
       [0.48472997, 0.02462871]])